# CASE 2: Inference Reproducibility with Independent  #

Because the number of classification folds across the 100 runs is too large, we only provide the Jupyter notebook for one fold here, namely the first fold of the reshuffling process. This notebook reproduces the zero setting for the Meta-learner model. To reproduce the other results, simply follow the same workflow and continue running the remaining folds.

## classification

### PanPep

##### majority

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_majority_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"  #Blank file
!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --support_dir "../data/support/majority" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "CASE2_majority"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: LLAGIGTVPI on device: cuda:0
Positive TCRs: 324, Negative TCRs: 9552
Total batches: 1
Loading support data from: ../data/support/majority
Loading support data for peptide: LLAGIGTVPI
Query data size: 9876

Processing batch 1/1 for peptide LLAGIGTVPI - 2026-03-16 23:29:01
Finetuning model...
batch processing time: 13.41s, Progress: 100.0%
Successfully wrote 9876 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE2_majority/LLAGIGTVPI.parquet

Peptide LLAGIGTVPI processing time: 13.46s

Processing peptide: TTDPSFLGRY on device: cuda:0
Positive TCRs: 166, Negative TCRs: 9552
Total batches: 1
Loading support data from: ../data/support/majority
Loading support data for peptide: TTDPSFLGRY
Query data size: 9718

Processing batch 1/1 for peptide TTDPSFLGRY - 2026-03-16 23:29:14
Finetuning model...
batch processing time: 12.91s, Progress: 100.0%
Successfully wrote 9718 records to /mnt/d/PanPep_Reu

In [14]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE2_majority"
CSV_PATH = "../data/fig4/reshuffling_majority_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE2_majority_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] ATDALMTGF: 52 rows, 52 scored
[OK] LLAGIGTVPI: 324 rows, 324 scored
[OK] LTDEMIAQY: 52 rows, 52 scored
[OK] TTDPSFLGRY: 166 rows, 166 scored

Saved to ./result/CASE2_majority_scored.csv
Total rows: 594, scored: 594


In [16]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE2_majority_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [17]:
import pandas as pd

df = pd.read_csv("./CASE2_majority_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE2_majority_scored.csv,0.544899,0.565792,success
1,./result,[FOLDER_SUMMARY],0.544899,0.565792,success (averaged over 1 files)


##### few-shot

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_few_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_few_shot.py \
    --mode single \
    --model default\
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --support_dir "../data/support/few" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "CASE2_few"

Using 1 GPUs: [0]
Using model configuration: default
Inference mode: single
support_dir enabled: will load k-shot data from ../data/support/few

Processing peptide: NTNSSPDDQIGYY on device: cuda:0
Positive TCRs: 10, Negative TCRs: 9552
Total batches: 1
Loading k-shot data from: ../data/support/few
Query data size: 9562

Processing batch 1/1 for peptide NTNSSPDDQIGYY - 2026-03-16 23:32:41
Finetuning model...
batch processing time: 1.09s, Progress: 100.0%
Successfully wrote 9562 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE2_few/NTNSSPDDQIGYY.parquet

Total time for peptide NTNSSPDDQIGYY: 1.14s

Processing peptide: RLLPLLALLAL on device: cuda:0
Positive TCRs: 26, Negative TCRs: 9552
Total batches: 1
Loading k-shot data from: ../data/support/few
Query data size: 9578

Processing batch 1/1 for peptide RLLPLLALLAL - 2026-03-16 23:32:42
Finetuning model...
batch processing time: 0.62s, Progress: 100.0%
Successfully wrote 9578 records to /mnt/d/PanPep_Reusability

In [22]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE2_few"
CSV_PATH = "../data/fig4/reshuffling_few_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE2_few_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AIFYLITPV: 18 rows, 18 scored
[OK] AIIRILQQL: 12 rows, 12 scored
[OK] ALASCMGLIY: 8 rows, 8 scored
[OK] ALDIEIATY: 8 rows, 8 scored
[OK] ALSKGVHFV: 70 rows, 70 scored
[OK] ALWEIQQVV: 68 rows, 68 scored
[OK] ALWMRLLPL: 28 rows, 28 scored
[OK] AMFWSVPTV: 156 rows, 156 scored
[OK] CLAVHECFV: 16 rows, 16 scored
[OK] CLNEYHLFL: 16 rows, 16 scored
[OK] DTDFVNEFY: 12 rows, 12 scored
[OK] ELAGIGLTV: 16 rows, 16 scored
[OK] ELTSMKYFV: 6 rows, 6 scored
[OK] FIAGLIAIV: 10 rows, 10 scored
[OK] FLAHIQWMV: 14 rows, 14 scored
[OK] FLALCADSI: 10 rows, 10 scored
[OK] FLCWHTNCY: 6 rows, 6 scored
[OK] FLGKIWPSYK: 16 rows, 16 scored
[OK] FLHVTYVPA: 6 rows, 6 scored
[OK] FLLNKEMYL: 14 rows, 14 scored
[OK] FLNGSCGSV: 6 rows, 6 scored
[OK] FLNRFTTTL: 8 rows, 8 scored
[OK] FLPGVYSVI: 12 rows, 12 scored
[OK] FLPRVFSAV: 12 rows, 12 scored
[OK] FLTENLLLY: 8 rows, 8 scored
[OK] FLVLLPLVS: 6 rows, 6 scored
[OK] FLWSVFWLI: 40 rows, 40 scored
[OK] FTSDYYQLY: 76 rows, 76 scored
[OK] FTVLCLTPV: 18 rows, 18 scored

In [25]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE2_few_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [26]:
import pandas as pd

df = pd.read_csv("./CASE2_few_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE2_few_scored.csv,0.52439,0.51448,success
1,./result,[FOLDER_SUMMARY],0.52439,0.51448,success (averaged over 1 files)


#### zero-shot

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_zero_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_zero_shot.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --result_dir "CASE2_zero"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --model default

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: ALWMRLLPLL on device: cuda:0

Processing batch 1/1 for peptide ALWMRLLPLL - 2026-03-16 23:36:22
batch processing time: 1.04s, Progress: 100.0%
Successfully wrote 9554 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE2_zero/ALWMRLLPLL.parquet

Peptide ALWMRLLPLL processing time: 1.07s

Processing peptide: CTDDNALAYY on device: cuda:0

Processing batch 1/1 for peptide CTDDNALAYY - 2026-03-16 23:36:23
batch processing time: 0.63s, Progress: 100.0%
Successfully wrote 9558 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE2_zero/CTDDNALAYY.parquet

Peptide CTDDNALAYY processing time: 0.65s

Processing peptide: EIFDRYGEEV on device: cuda:0

Processing batch 1/1 for peptide EIFDRYGEEV - 2026-03-16 23:36:24
batch processing time: 0.61s, Progress: 100.0%
Successfully wrote 9556 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE2_zero/EIFDRYGEEV.pa

In [29]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE2_zero"
CSV_PATH = "../data/fig4/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE1_zero_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AALPILFQV: 2 rows, 2 scored
[OK] AFLLFLVLI: 2 rows, 2 scored
[OK] ALASCMGLI: 2 rows, 2 scored
[OK] ALLPGLPAA: 8 rows, 8 scored
[OK] ALNLGETFV: 4 rows, 4 scored
[OK] ALNNIINNA: 4 rows, 4 scored
[OK] ALPETTADI: 4 rows, 4 scored
[OK] ALWMRLLPLL: 2 rows, 2 scored
[OK] ALYYPSARI: 4 rows, 4 scored
[OK] AMDEFIERY: 4 rows, 4 scored
[OK] AMQTMLFTM: 6 rows, 6 scored
[OK] ATSRMLSYY: 2 rows, 2 scored
[OK] ATSRTLSYY: 6 rows, 6 scored
[OK] AVLDMCASL: 2 rows, 2 scored
[OK] CLAYYFMRF: 6 rows, 6 scored
[OK] CLEASFNYL: 6 rows, 6 scored
[OK] CLGSLIYST: 2 rows, 2 scored
[OK] CTDDNALAYY: 6 rows, 6 scored
[OK] CTDDNALAY: 2 rows, 2 scored
[OK] DKDPQFKDN: 2 rows, 2 scored
[OK] DLTSFLLSL: 2 rows, 2 scored
[OK] DNVILLNKH: 6 rows, 6 scored
[OK] DRLNQLESK: 2 rows, 2 scored
[OK] DYKHYTPSF: 2 rows, 2 scored
[OK] DYTEISFML: 2 rows, 2 scored
[OK] EIFDRYGEEV: 4 rows, 4 scored
[OK] ETALALLLL: 2 rows, 2 scored
[OK] EWFLAYILF: 2 rows, 2 scored
[OK] FAMQMAYRF: 8 rows, 8 scored
[OK] FFSYFAVHF: 2 rows, 2 scored
[OK] FG

In [31]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE2_zero_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [32]:
import pandas as pd

df = pd.read_csv("./CASE2_zero_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE1_zero_scored.csv,0.48267,0.489076,success
1,./result,[FOLDER_SUMMARY],0.48267,0.489076,success (averaged over 1 files)


#### Meta-learner

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/fig4/reshuffling_zero_0.csv"
NEGATIVE_DATA = "../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_meta_learner.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --result_dir "CASE2_meta"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" 

Using 1 GPUs: [0]
GPU 0 will process 230 peptides with approximately 858 TCRs

Processing peptide: ALLPGLPAA on device: cuda:0
Positive TCRs: 8, Negative TCRs: 9553
Total batches: 1
Query data size: 9561

Processing batch 1/1 for peptide ALLPGLPAA - 2026-03-16 23:45:29
batch processing time: 0.87s, Progress: 100.0%
Successfully saved results to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE2_meta/ALLPGLPAA.parquet

Total time for peptide ALLPGLPAA: 0.98s

Processing peptide: FAMQMAYRF on device: cuda:0
Positive TCRs: 8, Negative TCRs: 9553
Total batches: 1
Query data size: 9561

Processing batch 1/1 for peptide FAMQMAYRF - 2026-03-16 23:45:30
batch processing time: 0.57s, Progress: 100.0%
Successfully saved results to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/CASE2_meta/FAMQMAYRF.parquet

Total time for peptide FAMQMAYRF: 0.66s

Processing peptide: FLFAVGFYL on device: cuda:0
Positive TCRs: 8, Negative TCRs: 9553
Total batches: 1
Query data size: 956

In [40]:
import os
import pandas as pd

PARQUET_DIR = "../inference/PanPep_Weight_Inference/CASE2_meta"
CSV_PATH = "../data/fig4/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE2_meta_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AALPILFQV: 2 rows, 2 scored
[OK] AFLLFLVLI: 2 rows, 2 scored
[OK] ALASCMGLI: 2 rows, 2 scored
[OK] ALLPGLPAA: 8 rows, 8 scored
[OK] ALNLGETFV: 4 rows, 4 scored
[OK] ALNNIINNA: 4 rows, 4 scored
[OK] ALPETTADI: 4 rows, 4 scored
[OK] ALWMRLLPLL: 2 rows, 2 scored
[OK] ALYYPSARI: 4 rows, 4 scored
[OK] AMDEFIERY: 4 rows, 4 scored
[OK] AMQTMLFTM: 6 rows, 6 scored
[OK] ATSRMLSYY: 2 rows, 2 scored
[OK] ATSRTLSYY: 6 rows, 6 scored
[OK] AVLDMCASL: 2 rows, 2 scored
[OK] CLAYYFMRF: 6 rows, 6 scored
[OK] CLEASFNYL: 6 rows, 6 scored
[OK] CLGSLIYST: 2 rows, 2 scored
[OK] CTDDNALAYY: 6 rows, 6 scored
[OK] CTDDNALAY: 2 rows, 2 scored
[OK] DKDPQFKDN: 2 rows, 2 scored
[OK] DLTSFLLSL: 2 rows, 2 scored
[OK] DNVILLNKH: 6 rows, 6 scored
[OK] DRLNQLESK: 2 rows, 2 scored
[OK] DYKHYTPSF: 2 rows, 2 scored
[OK] DYTEISFML: 2 rows, 2 scored
[OK] EIFDRYGEEV: 4 rows, 4 scored
[OK] ETALALLLL: 2 rows, 2 scored
[OK] EWFLAYILF: 2 rows, 2 scored
[OK] FAMQMAYRF: 8 rows, 8 scored
[OK] FFSYFAVHF: 2 rows, 2 scored
[OK] FG

In [41]:
!{sys.executable} ../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE2_meta_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [42]:
import pandas as pd

df = pd.read_csv("./CASE2_meta_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE2_meta_scored.csv,0.486413,0.485271,success
1,./result,[FOLDER_SUMMARY],0.486413,0.485271,success (averaged over 1 files)


### Other method

If you would like to reproduce other methods, you can use their web servers. For DLpTCR, you can use http://jianglab.org.cn/DLpTCR/ and select pTCRβ. For ERGO-II, you can use http://tcr2.cs.biu.ac.il/home and choose VDJdb mode.

## rank

For demonstration purposes, since running on the full background pool would be too time-consuming, we constructed a subset pool by randomly sampling 0.1% of the background library and used it for reproduction and result presentation. If you would like to reproduce the full original results, you can simply replace the subset background pool with the complete background library and rerun the same pipeline. In other words, the current setup is only for demonstration efficiency, while the full results can be recovered by using the original background pool.

In [ ]:
import sys

SCRIPT_DIR = "../inference/PanPep_Weight_Inference"
TEST_DATA = "../data/majority-fig3.csv"
NEGATIVE_DATA = "../data/Combined_library_sample_0.1pct.txt"

!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --support_dir "../data/support/majority" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "CASE2_majority_rank"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: LLAGIGTVPI on device: cuda:0
Positive TCRs: 806, Negative TCRs: 57103
Total batches: 6
Loading support data from: ../data/support/majority
Loading support data for peptide: LLAGIGTVPI
Query data size: 57265

Processing batch 1/6 for peptide LLAGIGTVPI - 2026-03-16 23:54:25
Finetuning model...
batch processing time: 15.23s, Progress: 16.7%

Processing batch 2/6 for peptide LLAGIGTVPI - 2026-03-16 23:54:40
Using finetuned model for inference...
batch processing time: 0.74s, Progress: 33.3%

Processing batch 3/6 for peptide LLAGIGTVPI - 2026-03-16 23:54:41
Using finetuned model for inference...
batch processing time: 0.71s, Progress: 50.0%

Processing batch 4/6 for peptide LLAGIGTVPI - 2026-03-16 23:54:41
Using finetuned model for inference...
batch processing time: 0.74s, Progress: 66.7%

Processing batch 5/6 for peptide LLAGIGTVPI - 2026-03-16 23:54:42
Using finetuned model for inference...
batch processing time: 

In [45]:
!{sys.executable} ../metric_calculation/sort.py \
                    --input_dir ../inference/PanPep_Weight_Inference/CASE2_majority_rank \
                    --output_dir ./CASE2_majority_sort

Found 4 files to process:
  CSV files: 0
  Parquet files: 4

Completed processing 4 files


In [ ]:
!{sys.executable} ../metric_calculation/bedroc.py \
                    -i ./CASE2_majority_sort/CASE2_majority_rank \
                    -o ./CASE2_majority_bedroc

2026-03-16 23:57:13,353 - INFO - Input directory: /mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE2_majority_rank
2026-03-16 23:57:13,354 - INFO - Output directory: ./CASE2_majority_bedroc
2026-03-16 23:57:13,354 - INFO - Number of processes: 8
Total: 4 files, To process: 4 files
Processing files:   0%|                                   | 0/4 [00:00<?, ?it/s]
File: TTDPSFLGRY_sorted.parquet

File: LLAGIGTVPI_sorted.parquet

File: LTDEMIAQY_sorted.parquet

File: ATDALMTGF_sorted.parquet
Processing files: 100%|███████████████████████████| 4/4 [00:00<00:00, 29.71it/s]
2026-03-16 23:57:13,579 - INFO - Successfully processed 4 files.
2026-03-16 23:57:13,579 - INFO - Detailed results stored in: ./CASE2_majority_bedroc/
2026-03-16 23:57:13,579 - INFO - Starting summary calculation...
2026-03-16 23:57:13,582 - INFO - Reading 4 result files...
Reading detailed results: 100%|██████████████████| 4/4 [00:00<00:00, 152.00it/s]
2026-03-16 23:57:13,628 - INFO - Summary saved to

In [48]:
import os
import glob
import pandas as pd

detailed_dir = "./CASE2_majority_bedroc"

# Print all CSV files in the detailed results directory
csv_files = sorted(glob.glob(os.path.join(detailed_dir, "*.csv")))

print(f"Found {len(csv_files)} CSV file(s) in: {detailed_dir}\n")

for file_path in csv_files:
    print("=" * 100)
    print(f"File: {file_path}")
    print("=" * 100)
    df = pd.read_csv(file_path)
    print(df)
    print("\n")

Found 5 CSV file(s) in: ./CASE2_majority_bedroc

File: ./CASE2_majority_bedroc/summary_by_directory.csv
                                           Directory  BEDROC_Mean  BEDROC_Std  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...     0.222439    0.108694   

   File_Count  
0           4  


File: ./CASE2_majority_bedroc/temp_ATDALMTGF_sorted.csv
                                           Directory  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...   

                   Filename  BEDROC_Score  Alpha  
0  ATDALMTGF_sorted.parquet      0.375515     20  


File: ./CASE2_majority_bedroc/temp_LLAGIGTVPI_sorted.csv
                                           Directory  \
0  /mnt/d/PanPep_Reusability/jupter/inference/Pan...   

                    Filename  BEDROC_Score  Alpha  
0  LLAGIGTVPI_sorted.parquet      0.157253     20  


File: ./CASE2_majority_bedroc/temp_LTDEMIAQY_sorted.csv
                                           Directory  \
0  /mnt/d/PanPep_Reusability/jupter/in

In [ ]:
!{sys.executable} "../metric_calculation/success_rate&hit_rate.py" \
    --root_dir "./CASE2_majority_sort/CASE2_majority_rank" \
    --output "CASE2_majority_sort" \
    --output_dir "./CASE2_majority_success"

2026-03-16 23:58:23,173 - INFO - Multiprocessing start method set to 'spawn'.
2026-03-16 23:58:23,252 - INFO - Configuration: ProcessingConfig(root_dir='/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE2_majority_rank', top_k_ratio=1, batch_size=150, num_gpus=1, output_file='CASE2_majority_sort', output_dir='./CASE2_majority_success', gpu_id=None)
2026-03-16 23:58:23,253 - INFO - Detected 1 GPU(s)
Processing directories: 100%|█████████████████████| 1/1 [00:02<00:00,  2.25s/it]
2026-03-16 23:58:25,529 - INFO - Combining 1 DataFrames...
2026-03-16 23:58:25,856 - INFO - Final results saved to CASE2_majority_success/CASE2_majority_sort.csv and CASE2_majority_success/CASE2_majority_sort.parquet


In [ ]:
!{sys.executable} "../metric_calculation/get_success_AUC.py" \
    --input "./CASE2_majority_success/CASE2_majority_sort.parquet" \
    --output "CASE2_majority_success_AUC" 

Found 1 directories.
1. '/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE2_majority_rank'
['/mnt/d/PanPep_Reusability/jupter/inference/PanPep_Weight_Inference/CASE2_majority_rank'] n=57265  Top_K range=(1, 57265)  y range=(0, 1)
/mnt/d/PanPep_Reusability/jupter/../metric_calculation/get_success_AUC.py:190: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area = float(np.trapz(ys, xs_norm))
  AUC@0.0100%: area=0
  AUC@0.1000%: area=6.11276e-06
  AUC@1.0000%: area=0.000438797
  AUC@5.0000%: area=0.00811599
  AUC@10.0000%: area=0.0244644
  AUC@20.0000%: area=0.0663292
  AUC@100.0000%: area=0.69559

Saved results to: CASE2_majority_success_AUC
